In [93]:
using Pkg                # Load Julia's built-in package manager module (Pkg)

Pkg.activate("..")       # Activate the Project.toml located one directory above
                         # (Allows notebooks/ to use the repository root environment)


using Revise             # Development tool that automatically reloads changes
# made to files in src/ without restarting Julia

using Horizons3DViewer2  # Load the custom package Horizons3DViewer2
# (This loads src/Horizons3DViewer2.jl and its modules)

using AstroTime, Dates   # Time systems and date handling utilities
using GLMakie            # High-performance interactive visualization backend
using ProgressMeter      # Progress bar utilities for long computations
using LinearAlgebra      # Standard linear algebra operations (norm, dot, etc.)

  Activating project at `~/work/Horizons3DViewer2`


In [94]:
using HTTP          # HTTPリクエストを送信するためのパッケージ
using JSON3         # JSONデータをパース（読み込み）するためのパッケージ
using Dates         # 日付・時刻を扱うための標準ライブラリ

const SSC_BASE = "https://sscweb.gsfc.nasa.gov/WS/sscr/2"  
# NASA SSCWeb (Satellite Situation Center Web) API のベースURL

function ssc_time(dt::DateTime)
    # DateTimeをSSC API用の文字列形式に変換する関数
    
    s = Dates.format(dt, dateformat"yyyymmddHHMMSS")
    # DateTimeを "yyyymmddHHMMSS" 形式の文字列に変換
    
    return s[1:8] * "T" * s[9:end] * "Z"
    # "YYYYMMDDTHHMMSSZ" 形式（ISO風）に整形して返す
end

function ssc_locations_json(observatory::String, t0::DateTime, t1::DateTime;
                            coord::String="geij2000", resolutionFactor::Int=1)
    # 指定した衛星(observatory)の位置データをSSC APIからJSON形式で取得する関数
    # coord: 座標系（デフォルトはGEI J2000）
    # resolutionFactor: データの時間分解能の係数

    tr = string(ssc_time(t0), ",", ssc_time(t1))
    # 開始時刻と終了時刻を「開始,終了」形式の文字列に変換

    url = "$SSC_BASE/locations/$observatory/$tr/$coord/?resolutionFactor=$resolutionFactor"
    # APIリクエスト用URLを構築

    println("URL = ", url)  
    # ★ デバッグ用：実際にアクセスしているURLを表示

    r = HTTP.get(url, ["Accept"=>"application/json", "User-Agent"=>"Julia SSC client"])
    # HTTP GETリクエストを送信
    # AcceptヘッダでJSON形式を要求
    # User-Agentを明示的に指定

    r.status == 200 || error("HTTP $(r.status): $(String(r.body))")
    # HTTPステータスコードが200（成功）でなければエラーを発生

    return JSON3.read(String(r.body))
    # レスポンスボディを文字列化し、JSONとしてパースして返す
end

t0 = DateTime(1981, 1, 1, 0, 0, 0)
# 取得開始時刻（1982年1月1日 00:00:00）

t1 = DateTime(1984, 1, 31, 23, 0, 0)
# 取得終了時刻（1984年1月1日 00:00:00）

data = ssc_locations_json("isee3", t0, t1; coord="geij2000", resolutionFactor=30)
# ISEE-3衛星の位置データをGEI J2000座標系で取得

println(typeof(data))
# 取得したデータの型を表示

URL = https://sscweb.gsfc.nasa.gov/WS/sscr/2/locations/isee3/19810101T000000Z,19840131T230000Z/geij2000/?resolutionFactor=30
JSON3.Array{Any, Base.CodeUnits{UInt8, String}, Vector{UInt64}}


In [95]:
# ------------------------------------------------------------
# SSCWebのレスポンスは
# ["クラス名", 実体]
# という2要素ベクトルでネストされている。
#
# その「型名ラッパー」を取り除くための関数。
# ------------------------------------------------------------
function unwrap(x)

    # x が AbstractVector かどうか確認
    if x isa AbstractVector

        # 要素数が2であるか確認
        if length(x) == 2

            # 1番目の要素が文字列（＝Javaの型名）か確認
            if x[1] isa AbstractString

                # ["型名", 実体] の形式なので、
                # 実体（2番目）を返す
                return x[2]

            end
        end
    end

    # 上記条件に当てはまらない場合は
    # そのまま x を返す
    return x
end


unwrap (generic function with 1 method)

In [96]:
# unwrap はあなたの定義を使用

resp   = unwrap(data)
result = unwrap(resp["Result"])

data2  = unwrap(result["Data"])
sat    = unwrap(data2[1])

coord  = unwrap(unwrap(sat["Coordinates"])[1])

# 位置
X = unwrap(coord["X"])
Y = unwrap(coord["Y"])
Z = unwrap(coord["Z"])
position = hcat(X, Y, Z)'   # 3×N 行列

# 角度
lat = unwrap(coord["Latitude"])
lon = unwrap(coord["Longitude"])

# 時刻（CoordinateDataではなくSatelliteData側）
t_raw = unwrap(sat["Time"])   # 各要素が ["javax.xml.datatype.XMLGregorianCalendar", "1982-..."] みたいな形
t_str = [ unwrap(x) for x in t_raw ]   # "1982-01-01T00:08:00.000+00:00" など

# 地心距離
r = unwrap(sat["RadialLength"])

println(length(position[1,:]))

for i in 1:3
    println(t_str[i])
end
println(t_str[end])



4504
1981-01-01T00:08:00.000+00:00
1981-01-01T06:08:00.000+00:00
1981-01-01T12:08:00.000+00:00
1984-01-31T18:08:00.000+00:00


In [97]:
using LinearAlgebra

ε = deg2rad(23.439291111)

R = [
    1  0           0
    0  cos(ε)  sin(ε)
    0 -sin(ε)  cos(ε)
]

position_ecl = R * position

3×4504 Matrix{Float64}:
     8.59573e5  850131.0       …   1.31144e6   1.33087e6   1.35024e6
    -1.35656e6      -1.3638e6     -6.51221e6  -6.56179e6  -6.61182e6
 71814.3         72601.7           3.03305e5   3.04157e5   3.04999e5

In [102]:

######################################
# Plotting area settings
######################################

set_theme!(theme_black())
update_theme!(fontsize= 18)

fig1 = Figure(size = (900, 600))

ax_ecliptic_s      = default_axis3(fig1[1,1]; scale=10e5,  title="",hide_decorations = false)

scatter!(ax_ecliptic_s, 0.0, 0.0, 0.0, markersize=15, color=:white)
lines!(ax_ecliptic_s, position_ecl[1:3, :], linewidth=1, color=:lime)

fig1


In [99]:
using Dates

# フォーマット指定
fmt = dateformat"yyyy-mm-ddTHH:MM:SS.sss+00:00"

# DateTimeベクトルへ変換
t = DateTime.(t_str, fmt)
Δt = diff(t)   # DateTime差分 → Period型（Millisecondなど）
println(minimum(Δt)/ 1000/60, maximum(Δt)/ 1000/60)
Δt_min = Dates.value.(Δt) ./ 1000/60



360 milliseconds360 milliseconds


4503-element Vector{Float64}:
 360.0
 360.0
 360.0
 360.0
 360.0
 360.0
 360.0
 360.0
 360.0
 360.0
   ⋮
 360.0
 360.0
 360.0
 360.0
 360.0
 360.0
 360.0
 360.0
 360.0

In [100]:
using AstroTime

# SSCの "…+00:00" を外して、from_utc が素直に読める形にする
# 例: "1982-01-01T00:08:00.000"
t_utc_str = replace.(t_str, r"\+00:00$" => "")

# UTC -> TDB（Epoch）へ（うるう秒も考慮される）
t_tdb = from_utc.(t_utc_str; scale=TDB)

# 確認
println(t_utc_str[1])
println(t_tdb[1])

1981-01-01T00:08:00.000
1981-01-01T00:08:51.183 TDB


In [101]:
using Dates

# ------------------------------------------------------------
# 入力：
#   t_str    : "1982-01-01T00:08:00.000+00:00" のような文字列配列
#   X, Y, Z  : SSCから取り出した各成分（同じ長さ）
#
# 出力：
#   t_new          : 等間隔に整列した DateTime ベクトル
#   position_new   : 補間後の 3×N 行列（各列が [x;y;z]）
#   gaps           : 飛び区間の情報（何番目→何番目が飛んだか等）
# ------------------------------------------------------------

# ① 文字列 -> DateTime (UTC)
fmt = dateformat"yyyy-mm-ddTHH:MM:SS.sss+00:00"
t = DateTime.(t_str, fmt)

# ② 位置 3×N
position = hcat(X, Y, Z)'  # 3×N

# ③ データ長チェック（念のため）
if length(t) != size(position, 2)
    error("length(t) != number of columns in position")
end

# ④ 時刻差分（ミリ秒）
Δt = diff(t)
Δt_ms = Dates.value.(Δt)  # millisecond の整数

# ⑤ 基準ステップを推定（最頻値：mode 相当を自前で計算）
counts = Dict{Int, Int}()
for v in Δt_ms
    if haskey(counts, v)
        counts[v] += 1
    else
        counts[v] = 1
    end
end

base_ms = first(keys(counts))
best_n  = -1
for (k, n) in counts
    if n > best_n
        best_n = n
        base_ms = k
    end
end

base_step = Millisecond(base_ms)
println("Base step ≈ ", base_step)

# ⑥ 「飛び」を検出（基準ステップの 1.5倍より大きい差分はギャップとみなす）
#    ※ 閾値は必要に応じて変更してください
gap_threshold_ms = round(Int, 1.5 * base_ms)

gaps = Vector{NamedTuple{(:i, :t0, :t1, :Δms, :missing_steps), Tuple{Int, DateTime, DateTime, Int, Int}}}()
for i in 1:length(Δt_ms)
    if Δt_ms[i] > gap_threshold_ms
        missing_steps = Int(round(Δt_ms[i] / base_ms)) - 1
        push!(gaps, (i=i, t0=t[i], t1=t[i+1], Δms=Δt_ms[i], missing_steps=missing_steps))
    end
end

println("Number of gaps: ", length(gaps))
for g in gaps
    println("gap at i=", g.i, " : ", g.t0, " -> ", g.t1,
            "  Δ=", Millisecond(g.Δms), "  missing_steps≈", g.missing_steps)
end

# ⑦ 新しい等間隔時刻グリッドを作成
#    最初〜最後まで base_step で刻む
t_new = collect(t[1]:base_step:t[end])

# ⑧ 線形補間で position を新グリッドへ写す（1次補間）
#    ここでは「各軸ごと」に補間します。
function interp1_linear(t_old::Vector{DateTime}, y_old::Vector{Float64}, t_query::Vector{DateTime})
    # 出力配列
    yq = Vector{Float64}(undef, length(t_query))

    # old のインデックス（単調増加前提）
    j = 1

    for i in 1:length(t_query)
        tq = t_query[i]

        # tq が末尾を超えないように調整（保険）
        if tq <= t_old[1]
            yq[i] = y_old[1]
        elseif tq >= t_old[end]
            yq[i] = y_old[end]
        else
            # tq が [t_old[j], t_old[j+1]] に入るまで j を進める
            while !(t_old[j] <= tq <= t_old[j+1])
                j += 1
            end

            t0 = t_old[j]
            t1 = t_old[j+1]
            y0 = y_old[j]
            y1 = y_old[j+1]

            # 線形補間係数（0〜1）
            α = Dates.value(tq - t0) / Dates.value(t1 - t0)

            yq[i] = (1 - α) * y0 + α * y1
        end
    end

    return yq
end

# ⑨ 各成分を補間
X_old = vec(position[1, :])
Y_old = vec(position[2, :])
Z_old = vec(position[3, :])

X_new = interp1_linear(t, X_old, t_new)
Y_new = interp1_linear(t, Y_old, t_new)
Z_new = interp1_linear(t, Z_old, t_new)

# ⑩ 3×N にまとめる
position_new = hcat(X_new, Y_new, Z_new)'  # 3×N

println("Old N = ", length(t), "  New N = ", length(t_new))

Base step ≈ 21600000 milliseconds
Number of gaps: 0
Old N = 4504  New N = 4504
